## Ch8. Exponential smoothing Forecasting: Principles & Practice (Python Edition) Extracted from: fpppy-08-exponential-smoothing.qmd

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## [Slide 3] Parameter Optimisation

In [2]:
algeria_economy = pd.read_csv("data/algeria_exports.csv", parse_dates=["ds"])


In [3]:
import numpy as np
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

sf = StatsForecast(
    models=[AutoETS(model="ANN", alias="SES")],
    freq="Y",
)
fc = sf.forecast(df=algeria_economy, h=5, level=[80, 95], fitted=True)
fitted_vals = sf.forecast_fitted_values()

Extract optimal parameters

In [4]:
sf.fit(df=algeria_economy)


StatsForecast(models=[SES])

In [5]:
ses = sf.fitted_[0, 0]
params = np.round(ses.model_["fit"].x, 4)
alpha, level_0 = params[0], params[1]
print(f"alpha = {alpha}, l0 = {level_0}")

alpha = 0.8401, l0 = 39.53


Example result: alpha = 0.84, l0 = 39.5

## [Slide 5] Holt's Linear Trend Method

In [6]:
aus_economy = pd.read_csv("data/aus_economy.csv", parse_dates=["ds"])


In [7]:
sf = StatsForecast(
    models=[AutoETS(model="AAN", alias="Holt")],
    freq="Y",
)
fc = sf.fit_predict(df=aus_economy, h=10)
holt = sf.fitted_[0, 0]
params = np.round(holt.model_["fit"].x, 4)

Example: alpha = 0.9999, beta* = 0.2001

## [Slide 7] Damped Trend Method

In [8]:
AutoETS(model="AAN", damped=True, phi=0.9, alias="Damped")

Damped

## [Slide 10] Holt-Winters' Multiplicative Method

In [9]:
aus_holidays = (
    pd.read_csv("data/tourism.csv", parse_dates=["ds"])
    .loc[lambda x: x["Purpose"] == "Holiday"]
    .groupby("ds", as_index=False)["y"].sum()
    .assign(unique_id="Holidays")
)


In [10]:
sf = StatsForecast(
    models=[
        AutoETS(season_length=4, model="AAA", alias="Additive"),
        AutoETS(season_length=4, model="MAM", alias="Multiplicative"),
    ],
    freq="Q",
)
fc = sf.forecast(df=aus_holidays, h=12, fitted=True)

## [Slide 17] Automatic Model Selection: AIC

In [11]:
sf = StatsForecast(models=[AutoETS(season_length=4)], freq="Q")
sf.fit(aus_holidays)
autoets = sf.fitted_[0, 0]
print(autoets.model_["method"])   # e.g. 'ETS(M,N,M)'
print(np.round(autoets.model_["fit"].x, 4))

ETS(M,N,M)
[3.6120000e-01 1.0000000e-04 9.7877911e+03 9.4320000e-01 9.2680000e-01
 9.6830000e-01]


## [Slide 19] Point Forecasts from ETS

In [12]:
sf = StatsForecast(
    models=[AutoETS(season_length=4)],
    freq="Q",
)
fc = sf.forecast(df=aus_holidays, h=8, level=[80, 95])

## [Slide 23] Summary: ETS vs.\ Benchmark

In [13]:
www_usage = pd.read_csv("data/www_usage.csv")


In [14]:
sf = StatsForecast(models=[AutoETS(season_length=1)], freq=1)


In [15]:
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import rmse, mae, mape
from functools import partial
from utilsforecast.losses import mase

cv = sf.cross_validation(df=www_usage, h=1, n_windows=90, step_size=1)
result_df = evaluate(
    cv.drop("cutoff", axis=1),
    metrics=[rmse, mae, mape, partial(mase, seasonality=1)],
    train_df=www_usage,
)